# Midterm Discourse: K-Means Clustering, Step by Step
## DATA 78000 · AI and Machine Learning for Artists and Humanists

**Research question:** How does the language used in HasanAbi and Fox News comments change before, on, and after November 3, 2026?

This follows Professor Mandiberg's class demo on Iris: **load data → make numeric features → elbow method → fit K-Means → label rows → inspect centroids → visualize → interpret**.

The earlier production script lives at scripts/analyze.py and updates the public site. **This is a teaching version; it does not update the live site.**

Class demo: https://github.com/mandiberg/DATA-78000/tree/main/data_clustering

**Privacy:** Individual comment text stays in the ignored data/private/ folder. The notebook has no saved outputs. Do not commit executed notebooks that expose comments.

## Before starting — Install and collect your own copy

Run from the project root:

    python -m pip install pandas numpy scikit-learn matplotlib
    python scripts/collect.py --target 150

The collector requires YOUTUBE_API_KEY in your environment. A successful GitHub Actions run does **not** leave a local CSV in the repo; the private comment text stays on its temporary runner.

In Colab, upload comments.csv privately to the temporary notebook session. For a class demonstration, keep individual comment outputs hidden.

## Step 1 — Import the Python libraries

These tools replace the Iris dataset's built-in numerical measurements with numerical representations of comment text.

In [ ]:
import html
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import silhouette_score

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
print("Imports complete.")

## Step 2 — Load and inspect the dataset

Each row is one comment. Its channel and phase describe where it came from; we do **not** use those labels to train K-Means. In the original pilot there were 900 comments, 150 per video.

In [ ]:
paths = [Path("data/private/comments.csv"),
         Path("../data/private/comments.csv"),
         Path("comments.csv")]
paths = [p for p in paths if p.exists()]
if not paths:
    raise FileNotFoundError(
        "Collect comments locally with scripts/collect.py, "
        "or upload private comments.csv to this notebook session."
    )
df = pd.read_csv(paths[0])
required = {"comment_id", "channel", "phase", "text"}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"Missing CSV columns: {sorted(missing)}")
df = df.drop_duplicates(subset="comment_id").dropna(subset=["text"]).copy()
print("Number of unique comments:", len(df))
print(pd.crosstab(df["channel"], df["phase"]).to_string())

## Step 3 — Clean the text

We remove links, decode HTML escapes, and lowercase. Like the existing script, comments shorter than three words are excluded. Write down how many are excluded: that choice can change the results.

In [ ]:
def clean_text(value):
    value = html.unescape(str(value)).lower()
    value = re.sub(r"https?://\S+|www\.\S+", " ", value)
    return re.sub(r"\s+", " ", value).strip()

df["text_clean"] = df["text"].apply(clean_text)
df["word_count"] = df["text_clean"].str.split().str.len()
eligible = df.loc[df["word_count"] >= 3].copy().reset_index(drop=True)
print("Eligible comments:", len(eligible))
print("Excluded short comments:", len(df) - len(eligible))
print(pd.crosstab(eligible["channel"], eligible["phase"]).to_string())
if len(eligible) < 12:
    raise ValueError("At least 12 eligible comments are needed.")

## Step 4 — Convert words into numeric features (TF-IDF)

The Iris demo uses four measurements; we use thousands of numerical word/phrase weights. One vectorizer across **all** channels/periods ensures a shared feature space. TF-IDF normally L2-normalizes each comment vector. The channel/period labels are not features.

In [ ]:
vectorizer = TfidfVectorizer(
    stop_words="english",
    min_df=2,
    max_df=0.90,
    ngram_range=(1, 2),
    sublinear_tf=True,
    max_features=3000
)
X = vectorizer.fit_transform(eligible["text_clean"])
feature_names = vectorizer.get_feature_names_out()
print("Comments x features:", X.shape)
print("Example features:", list(feature_names[:15]))
if X.shape[1] < 2:
    raise ValueError("Not enough distinct TF-IDF features.")

## Step 5 — Make the elbow plot

This closely follows the class Iris demo. Inertia measures within-cluster squared distances. It decreases as k increases; look for where improvements begin to level out. The supplemental cosine-silhouette value is a *different* diagnostic, not the objective K-Means optimizes. Neither proves a cluster is meaningful.

In [ ]:
k_values = list(range(1, min(10, len(eligible) - 1) + 1))
inertia = []
silhouette = []
for k in k_values:
    trial = KMeans(n_clusters=k, random_state=RANDOM_SEED, n_init=10)
    trial_labels = trial.fit_predict(X)
    inertia.append(trial.inertia_)
    if k >= 2 and len(set(trial_labels)) >= 2:
        value = silhouette_score(
            X, trial_labels, metric="cosine",
            sample_size=min(400, X.shape[0]), random_state=RANDOM_SEED
        )
    else:
        value = np.nan
    silhouette.append(value)
scores = pd.DataFrame({"k": k_values, "inertia": inertia,
                       "cosine_silhouette": silhouette})
print(scores.round(3).to_string(index=False))
plt.figure(figsize=(8, 4))
plt.plot(k_values, inertia, marker="o", linestyle="--")
plt.xticks(k_values)
plt.xlabel("Number of clusters, k")
plt.ylabel("Inertia")
plt.title("Elbow method: midterm comments")
plt.grid(alpha=0.2)
plt.show()

## Step 6 — Choose k, fit K-Means and attach labels

The original script chose k=6 automatically based on dataset size. For this classroom exercise, **start with 4** and compare it to 3, 5, and 6. Justify your final choice using the elbow plot **and** human interpretation. Cluster IDs are arbitrary, not ideology labels.

In [ ]:
CHOSEN_K = 4   # Change this after reading the plot and cluster contents.
final_model = KMeans(n_clusters=CHOSEN_K, random_state=RANDOM_SEED, n_init=10)
cluster_labels = final_model.fit_predict(X)
eligible["cluster"] = cluster_labels   # Like the class assign-back-to-dataframe example.
print("Final inertia:", round(final_model.inertia_, 2))
print("Cluster sizes:")
print(eligible["cluster"].value_counts().sort_index().to_string())

## Step 7 — Examine centroids, like the Iris exercise

A centroid here is a vector of TF-IDF weights, not flower measurements. Print its highest-weight words. Give clusters descriptive labels **only after** checking their actual content.

In [ ]:
centroids = final_model.cluster_centers_
rows = []
for cluster_id in range(CHOSEN_K):
    indices = np.argsort(centroids[cluster_id])[::-1][:10]
    rows.append({
        "cluster": cluster_id,
        "n_comments": int((eligible["cluster"] == cluster_id).sum()),
        "top_words": ", ".join(feature_names[i] for i in indices)
    })
centroid_table = pd.DataFrame(rows)
print(centroid_table.to_string(index=False))

## Step 8 — Inspect examples privately

This mirrors the final stage of the Iris demo that prints points belonging to each cluster. Here we identify five comments closest to each centroid. Keep the text private: set SHOW_PRIVATE_EXAMPLES to True only in your own session, and remove outputs before sharing.

In [ ]:
SHOW_PRIVATE_EXAMPLES = False
distances = final_model.transform(X)
for cluster_id in range(CHOSEN_K):
    members = np.where(cluster_labels == cluster_id)[0]
    nearest = sorted(members, key=lambda i: distances[i, cluster_id])[:5]
    print(f"\nCluster {cluster_id}: {len(members)} eligible comments")
    print(centroid_table.loc[cluster_id, "top_words"])
    if SHOW_PRIVATE_EXAMPLES:
        for row_index in nearest:
            item = eligible.iloc[row_index]
            print(f"  {item['channel']} / {item['phase']}: {item['text'][:180]}")
    else:
        print("  Examples hidden; enable private review to assess interpretability.")

## Step 9 — Compare group percentages rather than raw counts

Now use the channel and phase metadata **after** clustering. Percentage denominators are the eligible comments per channel/phase, not necessarily 150, because the short-comment exclusion removed some rows. This is the core test of the research question.

In [ ]:
channels = ["HasanAbi", "Fox News"]
phases = ["before", "election", "after"]
index = pd.MultiIndex.from_product([channels, phases], names=["channel", "phase"])
counts = pd.crosstab(
    index=[eligible["channel"], eligible["phase"]],
    columns=eligible["cluster"]
).reindex(index, fill_value=0).reindex(columns=range(CHOSEN_K), fill_value=0)
percentages = counts.div(counts.sum(axis=1).replace(0, np.nan), axis=0).fillna(0) * 100
print("Percent of eligible comments per group:")
print(percentages.round(1).to_string())
plt.figure(figsize=(9, 5))
plt.imshow(percentages.values, aspect="auto", cmap="viridis", vmin=0, vmax=100)
plt.colorbar(label="% of eligible group")
plt.xticks(range(CHOSEN_K), [f"Cluster {i}" for i in range(CHOSEN_K)])
plt.yticks(range(len(index)), [f"{c} / {p}" for c, p in index])
plt.title("How cluster prevalence changes over the election")
plt.tight_layout()
plt.show()

## Step 10 — Plot two dimensions, as in the Iris demo

The demo plots two flower measurements. TF-IDF has thousands of dimensions, so here we project comments and centroids into two dimensions with TruncatedSVD. The projection loses information; it is **not** the space in which K-Means was trained.

In [ ]:
svd = TruncatedSVD(n_components=2, random_state=RANDOM_SEED)
points_2d = svd.fit_transform(X)
centers_2d = svd.transform(final_model.cluster_centers_)
plt.figure(figsize=(8, 6))
plt.scatter(points_2d[:, 0], points_2d[:, 1],
            c=cluster_labels, s=20, alpha=0.5, cmap="viridis")
plt.scatter(centers_2d[:, 0], centers_2d[:, 1],
            marker="X", c="red", s=150, label="Projected centroids")
plt.title("Comment clusters in a 2D projection")
plt.xlabel("SVD axis 1")
plt.ylabel("SVD axis 2")
plt.legend()
plt.show()
print("Fraction of variance explained by 2 axes:",
      f"{svd.explained_variance_ratio_.sum():.1%}")

## Step 11 — Repeat with other cluster counts

K-Means always assigns points to clusters, even when those clusters are incoherent. Inspect whether one large catch-all cluster dominates, as it did in the original six-cluster pilot.

In [ ]:
comparison = []
for k in [3, 4, 5, 6]:
    if k >= len(eligible):
        continue
    trial = KMeans(n_clusters=k, random_state=RANDOM_SEED, n_init=10)
    assignments = trial.fit_predict(X)
    sizes = pd.Series(assignments).value_counts()
    comparison.append({"k": k, "inertia": round(trial.inertia_, 2),
                       "largest_cluster_pct": round(sizes.max() / len(eligible) * 100, 1),
                       "smallest_cluster_n": int(sizes.min())})
print(pd.DataFrame(comparison).to_string(index=False))

## Step 12 — Interpretation / reflection (write this in your own words)

1. What changes in topic prevalence before, on, and after Election Day?
2. Which **specific words** characterize each cluster?
3. Does the elbow curve support your chosen k? Which k gives the clearest groups?
4. How many comments were excluded? Do the excluded rates vary by channel or period?
5. Did any cluster largely capture generic language, memes, or unrelated news instead of election narratives?
6. Did video selection or differing comment moderation create an alternative explanation?
7. What would you change before expanding to eight channels and 24 videos?

The previous k=6 run assigned **512 of 831** model-eligible comments to one generic cluster (61.6%). Treat this as an interpretability warning, **not as evidence** of an ideological group.

**Deliverable for class:** an elbow figure, a cluster-size table, a centroid/top-words table, a normalized channel-period heatmap, a projected scatterplot, and a discussion of meaningfulness/limitations.

Keep the existing scripts/analyze.py for the deployed site; the notebook is intentionally independent.